

## Cell 01 — Imports, Config & Environment

In [1]:
import os, time, json, hashlib, sqlite3, re, random
from datetime import datetime, timedelta
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

import gspread
from google.oauth2.service_account import Credentials

# ── Load credentials from .env ────────────────────────────────────────────────
load_dotenv()
LINKEDIN_EMAIL    = os.getenv("LINKEDIN_EMAIL")
LINKEDIN_PASSWORD = os.getenv("LINKEDIN_PASSWORD")
SHEET_NAME        = os.getenv("GOOGLE_SHEET_NAME", "Adya AI LinkedIn Mentions")
SA_JSON_PATH      = os.getenv("SERVICE_ACCOUNT_JSON", "service_account.json")

assert LINKEDIN_EMAIL,    "❌ LINKEDIN_EMAIL missing from .env"
assert LINKEDIN_PASSWORD, "❌ LINKEDIN_PASSWORD missing from .env"

# ── Scraper constants ─────────────────────────────────────────────────────────
SEARCH_TERMS   = ["Shayak Mazumder", "Adya AI"]
MONTHS_BACK    = 6
CUTOFF_DATE    = datetime.now() - timedelta(days=MONTHS_BACK * 30)
MAX_SCROLLS    = 40          
SCROLL_PAUSE   = (2.5, 5.0)  

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH   = OUTPUT_DIR / "linkedin_posts.db"
JSON_PATH = OUTPUT_DIR / "linkedin_mentions.json"

# ── Schema ────────────────────────────────────────────────────────────────────
POST_FIELDS = [
    "post_id", "post_text", "author_name", "author_headline",
    "author_profile_url", "author_followers", "post_url",
    "date_posted", "date_parsed", "reactions", "comments",
    "reposts", "search_term", "extracted_at"
]

print("✅ Cell 01 complete")
print(f"   Searching for : {SEARCH_TERMS}")
print(f"   Date cutoff   : {CUTOFF_DATE.strftime('%Y-%m-%d')}")
print(f"   Output folder : {OUTPUT_DIR.resolve()}")

✅ Cell 01 complete
   Searching for : ['Shayak Mazumder', 'Adya AI']
   Date cutoff   : 2025-09-13
   Output folder : D:\Adya Techology\LinkedinScrapper\output


## Cell 02 — Helper Functions

In [2]:
def make_post_id(post_url: str, author: str) -> str:
    """Deterministic dedup key from URL + author."""
    raw = f"{post_url.strip()}|{author.strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:14]


def parse_relative_date(text: str) -> str:
    """
    Convert LinkedIn relative dates to ISO format.
    Handles: '2h', '3d', '1w', '2mo', 'Just now', ISO strings.
    """
    if not text:
        return ""
    text = text.strip().lower()
    now  = datetime.now()

    if "just now" in text or "now" == text:
        return now.strftime("%Y-%m-%d")

    # patterns: 2h / 3d / 1w / 2mo / 1yr
    m = re.search(r"(\d+)\s*(h|d|w|mo|yr)", text)
    if m:
        n, unit = int(m.group(1)), m.group(2)
        delta = {"h": timedelta(hours=n), "d": timedelta(days=n),
                 "w": timedelta(weeks=n), "mo": timedelta(days=n*30),
                 "yr": timedelta(days=n*365)}.get(unit, timedelta(0))
        return (now - delta).strftime("%Y-%m-%d")

    # try ISO parse
    for fmt in ("%Y-%m-%d", "%Y-%m-%dT%H:%M:%S", "%B %d, %Y", "%b %d, %Y"):
        try:
            return datetime.strptime(text[:10], fmt[:len(text[:10])]).strftime("%Y-%m-%d")
        except ValueError:
            pass

    return text  # return raw if unparseable


def extract_number(text: str) -> int:
    """Pull first integer from strings like '1,234 reactions' → 1234."""
    if not text:
        return 0
    text = text.replace(",", "").replace(".", "")
    m = re.search(r"\d+", text)
    return int(m.group()) if m else 0


def clean(text: str) -> str:
    """Collapse whitespace."""
    return " ".join(str(text).split()).strip() if text else ""


def random_sleep(lo: float = SCROLL_PAUSE[0], hi: float = SCROLL_PAUSE[1]):
    time.sleep(random.uniform(lo, hi))


def is_within_window(date_str: str) -> bool:
    """True if date_str is within the last MONTHS_BACK months."""
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        return dt >= CUTOFF_DATE
    except Exception:
        return True  # fail-open: keep post if date is unparseable


print("✅ Cell 02 complete — helpers loaded")

✅ Cell 02 complete — helpers loaded


## Cell 03 — Launch Chrome & Login to LinkedIn

In [3]:
HEADLESS = False

def build_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    # Anti-detection options
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--window-size=1440,900")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=opts)
    # Mask webdriver property
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


def linkedin_login(driver: webdriver.Chrome) -> bool:
    wait = WebDriverWait(driver, 20)
    print("  → Opening LinkedIn login page...")
    driver.get("https://www.linkedin.com/login")
    random_sleep(2, 4)

    try:
        email_field = wait.until(EC.presence_of_element_located((By.ID, "username")))
        email_field.clear()
        # Added Delay to type like a human
        for ch in LINKEDIN_EMAIL:
            email_field.send_keys(ch)
            time.sleep(random.uniform(0.04, 0.12))

        pass_field = driver.find_element(By.ID, "password")
        pass_field.clear()
        for ch in LINKEDIN_PASSWORD:
            pass_field.send_keys(ch)
            time.sleep(random.uniform(0.04, 0.12))

        random_sleep(0.5, 1.5)
        driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()
        random_sleep(4, 6)

        # Check for successful login
        if "/feed" in driver.current_url or "/mynetwork" in driver.current_url:
            print("  ✅ Logged in successfully!")
            return True
        elif "checkpoint" in driver.current_url or "challenge" in driver.current_url:
            print("  ⚠️  LinkedIn is asking for verification.")
            print("      Complete it manually in the browser, then re-run from Cell 04.")
            return False
        else:
            print(f"  ❌ Unexpected URL after login: {driver.current_url}")
            return False

    except TimeoutException:
        print("  ❌ Login form not found — check your ChromeDriver version.")
        return False


# ── Launch & Login ────────────────────────────────────────────────────────────
print("🚀 Launching Chrome...")
driver = build_driver(headless=HEADLESS)
logged_in = linkedin_login(driver)

if not logged_in:
    print("\n⛔ Cannot continue without login. Fix the issue then re-run this cell.")
else:
    print(f"\n✅ Cell 03 complete — driver ready @ {driver.current_url}")

🚀 Launching Chrome...
  → Opening LinkedIn login page...
  ✅ Logged in successfully!

✅ Cell 03 complete — driver ready @ https://www.linkedin.com/feed/


## Cell 04 — Search LinkedIn & Scroll to Load Posts

In [4]:
import urllib.parse

raw_html_snapshots = []   # list of (term, html_string) tuples


def build_search_url(term: str) -> str:
    """
    LinkedIn content search URL sorted by date.
    datePosted=past-month is the tightest LinkedIn natively offers;
    we apply our own 6-month filter in the parsing step.
    """
    encoded = urllib.parse.quote(f'"{term}"')  
    return (
        f"https://www.linkedin.com/search/results/content/"
        f"?keywords={encoded}&sortBy=date_posted&f_TPR=r15552000"  
    )


def scroll_and_collect(driver: webdriver.Chrome, term: str) -> list[str]:
    """
    Navigate to search results for `term`, scroll until:
      - We hit MAX_SCROLLS, OR
      - The oldest visible post is before CUTOFF_DATE, OR
      - LinkedIn shows 'No more results'
    Returns list of page_source HTML strings (one per scroll position).
    """
    url = build_search_url(term)
    print(f"\n🔍 Searching: '{term}'")
    print(f"   URL: {url}")
    driver.get(url)
    random_sleep(3, 5)

    # Check we got results
    if "authwall" in driver.current_url:
        raise RuntimeError("⛔ Hit LinkedIn auth wall — session may have expired. Re-run Cell 03.")

    snapshots = []
    last_height = 0
    no_new_count = 0

    for scroll_n in range(MAX_SCROLLS):
        # Capture current HTML
        snapshots.append(driver.page_source)

        # Scroll down
        driver.execute_script("window.scrollBy(0, 1800);")
        random_sleep(*SCROLL_PAUSE)

        new_height = driver.execute_script("return document.body.scrollHeight")

        # Detect stall — no new content loaded
        if new_height == last_height:
            no_new_count += 1
            if no_new_count >= 3:
                print(f"   ↳ No new content after 3 attempts — stopping at scroll {scroll_n+1}")
                break
        else:
            no_new_count = 0
        last_height = new_height

        # Check if 'No results found' or 'See more results' is gone
        soup_check = BeautifulSoup(driver.page_source, "html.parser")
        no_results = soup_check.find(string=re.compile(r"No results found", re.I))
        if no_results:
            print("   ↳ LinkedIn says 'No results found' — done.")
            break

        # Optionally click 'Show more results' button if present
        try:
            more_btn = driver.find_element(
                By.XPATH,
                "//button[contains(., 'Show more results') or contains(., 'See more')]"
            )
            driver.execute_script("arguments[0].click();", more_btn)
            random_sleep(2, 4)
        except NoSuchElementException:
            pass

        print(f"   Scroll {scroll_n+1:02d} — page height: {new_height}px", end="\r")

    print(f"\n   ✅ Collected {len(snapshots)} HTML snapshots for '{term}'")
    return snapshots


# ── Run for both search terms ─────────────────────────────────────────────────
for term in SEARCH_TERMS:
    snaps = scroll_and_collect(driver, term)
    for snap in snaps:
        raw_html_snapshots.append((term, snap))
    random_sleep(5, 10)  # pause between different searches

print(f"\n✅ Cell 04 complete — {len(raw_html_snapshots)} total snapshots collected")


🔍 Searching: 'Shayak Mazumder'
   URL: https://www.linkedin.com/search/results/content/?keywords=%22Shayak%20Mazumder%22&sortBy=date_posted&f_TPR=r15552000
   Scroll 40 — page height: 76311px
   ✅ Collected 40 HTML snapshots for 'Shayak Mazumder'

🔍 Searching: 'Adya AI'
   URL: https://www.linkedin.com/search/results/content/?keywords=%22Adya%20AI%22&sortBy=date_posted&f_TPR=r15552000
   Scroll 40 — page height: 76650px
   ✅ Collected 40 HTML snapshots for 'Adya AI'

✅ Cell 04 complete — 80 total snapshots collected


## Cell 05 — BeautifulSoup Parser: Extract All Fields

In [5]:
def parse_post_card(card, term: str) -> dict | None:
    """
    Extract all fields from a single LinkedIn post card element.
    LinkedIn's DOM changes over time — we try multiple selectors
    with fallbacks so the scraper stays robust.
    """
    post = {}

    # ── Post text ─────────────────────────────────────────────────────────────
    text_selectors = [
        "div.feed-shared-update-v2__description span.break-words",
        "div.update-components-text span.break-words",
        "span[dir='ltr']",
        "div.feed-shared-text"
    ]
    post["post_text"] = ""
    for sel in text_selectors:
        el = card.select_one(sel)
        if el:
            post["post_text"] = clean(el.get_text(separator=" "))
            break

    if not post["post_text"]:
        return None  # skip cards with no text

    # ── Author name ───────────────────────────────────────────────────────────
    author_selectors = [
        "span.update-components-actor__name span[aria-hidden='true']",
        "span.feed-shared-actor__name",
        "a.update-components-actor__meta-link span[aria-hidden='true']",
        "span.app-aware-link span[aria-hidden='true']"
    ]
    post["author_name"] = ""
    for sel in author_selectors:
        el = card.select_one(sel)
        if el:
            post["author_name"] = clean(el.get_text())
            break

    # ── Author headline ───────────────────────────────────────────────────────
    headline_selectors = [
        "span.update-components-actor__description",
        "span.feed-shared-actor__description",
        "div.update-components-actor__meta span:not([aria-hidden])"
    ]
    post["author_headline"] = ""
    for sel in headline_selectors:
        el = card.select_one(sel)
        if el:
            post["author_headline"] = clean(el.get_text())
            break

    # ── Author profile URL ────────────────────────────────────────────────────
    post["author_profile_url"] = ""
    for a in card.select("a[href*='/in/']"):
        href = a.get("href", "")
        if "/in/" in href:
            # Strip query params
            post["author_profile_url"] = "https://www.linkedin.com" + href.split("?")[0] \
                if href.startswith("/") else href.split("?")[0]
            break

    # ── Author followers (if visible) ─────────────────────────────────────────
    post["author_followers"] = ""
    for el in card.select("span.update-components-actor__supplementary-actor-info"):
        t = clean(el.get_text())
        if "follower" in t.lower() or "connection" in t.lower():
            post["author_followers"] = t
            break

    # ── Post URL ──────────────────────────────────────────────────────────────
    post["post_url"] = ""
    url_selectors = [
        "a[href*='/posts/']",
        "a[href*='/feed/update/']",
        "a.app-aware-link[href*='activity']"
    ]
    for sel in url_selectors:
        el = card.select_one(sel)
        if el:
            href = el.get("href", "")
            post["post_url"] = href.split("?")[0] if href.startswith("http") \
                else "https://www.linkedin.com" + href.split("?")[0]
            break

    # ── Date posted ───────────────────────────────────────────────────────────
    post["date_posted"] = ""
    post["date_parsed"] = ""
    for sel in ["span.update-components-actor__sub-description",
                "span.feed-shared-actor__sub-description",
                "time", "a[aria-label*='ago']"]:
        el = card.select_one(sel)
        if el:
            raw_date = clean(el.get("datetime") or el.get("aria-label") or el.get_text())
            post["date_posted"] = raw_date
            post["date_parsed"] = parse_relative_date(raw_date)
            break

    # ── Engagement counts ─────────────────────────────────────────────────────
    # Reactions
    post["reactions"] = 0
    for sel in [
        "button[aria-label*='reaction'] span",
        "span.social-details-social-counts__reactions-count",
        "button.feed-shared-social-actions__count-reaction span",
        "li.social-details-social-counts__reactions button"
    ]:
        el = card.select_one(sel)
        if el:
            post["reactions"] = extract_number(el.get_text())
            break

    # Comments
    post["comments"] = 0
    for sel in [
        "button[aria-label*='comment'] span",
        "li.social-details-social-counts__comments button span",
        "a.social-details-social-counts__count-value"
    ]:
        el = card.select_one(sel)
        if el:
            t = el.get_text()
            if "comment" in (el.get("aria-label", "") + t).lower() or True:
                val = extract_number(t)
                if val > 0:
                    post["comments"] = val
                    break

    # Reposts
    post["reposts"] = 0
    for sel in [
        "button[aria-label*='repost'] span",
        "li.social-details-social-counts__reshares button span",
        "button.feed-shared-social-actions__count-reposts span"
    ]:
        el = card.select_one(sel)
        if el:
            post["reposts"] = extract_number(el.get_text())
            break

    # ── Metadata ──────────────────────────────────────────────────────────────
    post["search_term"]   = term
    post["extracted_at"]  = datetime.now().isoformat()
    post["post_id"]       = make_post_id(post["post_url"], post["author_name"])

    return post


def parse_all_snapshots(snapshots: list[tuple]) -> list[dict]:
    """Parse every HTML snapshot and return flat list of posts."""
    all_posts = []
    seen_ids  = set()

    for term, html in snapshots:
        soup = BeautifulSoup(html, "html.parser")

        # LinkedIn renders post cards with these container classes (multiple fallbacks)
        card_selectors = [
            "div.feed-shared-update-v2",
            "div.update-components-actor",
            "li.reusable-search__result-container",
            "div[data-urn]"
        ]

        cards = []
        for sel in card_selectors:
            cards = soup.select(sel)
            if cards:
                break

        for card in cards:
            post = parse_post_card(card, term)
            if not post:
                continue
            if post["post_id"] in seen_ids:
                continue
            seen_ids.add(post["post_id"])
            all_posts.append(post)

    return all_posts


# ── Parse ─────────────────────────────────────────────────────────────────────
print("⚙️  Parsing HTML snapshots...")
raw_posts = parse_all_snapshots(raw_html_snapshots)
print(f"✅ Cell 05 complete — extracted {len(raw_posts)} raw posts")

# Quick preview
if raw_posts:
    print("\nSample post:")
    for k, v in list(raw_posts[0].items())[:6]:
        print(f"  {k:25s}: {str(v)[:80]}")

⚙️  Parsing HTML snapshots...
✅ Cell 05 complete — extracted 156 raw posts

Sample post:
  post_text                : Grateful for another inspiring learning experience at KIT – Coimbatore! Today we
  author_name              : Neha Anish
  author_headline          : Student at KIT-Kalaignarkarunanidhi Institute of TechnologyStudent at KIT-Kalaig
  author_profile_url       : https://www.linkedin.com/in/nehaanish21
  author_followers         : 
  post_url                 : 


## Cell 06 — Pandas: Clean, Deduplicate & Date-Filter

In [6]:
df = pd.DataFrame(raw_posts, columns=POST_FIELDS)

print(f"Before cleaning : {len(df)} rows")

# 1. Drop rows with no post text
df = df[df["post_text"].str.strip().astype(bool)]

# 2. Deduplicate on post_id
df = df.drop_duplicates(subset=["post_id"], keep="first")

# 3. Deduplicate on post_url (secondary key)
df = df[df["post_url"] != ""]   # remove rows with no URL (they can't be deduped reliably)
df = df.drop_duplicates(subset=["post_url"], keep="first")

# 4. Apply 6-month date filter
df["_keep"] = df["date_parsed"].apply(is_within_window)
df = df[df["_keep"]].drop(columns=["_keep"])

# 5. Cast engagement columns to int
for col in ["reactions", "comments", "reposts"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# 6. Sort chronologically — newest first
df["_sort_date"] = pd.to_datetime(df["date_parsed"], errors="coerce")
df = df.sort_values("_sort_date", ascending=False).drop(columns=["_sort_date"])
df = df.reset_index(drop=True)

print(f"After cleaning  : {len(df)} rows")
print(f"\nBreakdown by search term:")
print(df["search_term"].value_counts().to_string())
print(f"\nDate range: {df['date_parsed'].min()}  →  {df['date_parsed'].max()}")
print(f"\n✅ Cell 06 complete")
df.head(5)

Before cleaning : 156 rows
After cleaning  : 20 rows

Breakdown by search term:
search_term
Adya AI            10
Shayak Mazumder    10

Date range: 2025-10-13  →  2026-03-10

✅ Cell 06 complete


,post_id,post_text,author_name,author_headline,author_profile_url,author_followers,post_url,date_posted,date_parsed,reactions,comments,reposts,search_term,extracted_at
0,bb8a401c612d72,Adya AI is hiring for interns !!,Mahesh siddamsetty,Product Management Leader - AI | B2B SaaS (0 t...,https://www.linkedin.com/in/maheshsiddamsetty,,https://www.linkedin.com/feed/update/urn:li:ac...,2d • 2 days ago • Visible to anyone on or off ...,2026-03-10,6,0,0,Adya AI,2026-03-12T23:02:58.065412
1,6d7428e5982b3e,"When Microsoft chooses to tell your story, it ...",Adya,"2,594 followers2,594 followers",https://www.linkedin.com/in/smhaaz/,,https://www.linkedin.com/feed/update/urn:li:ac...,1w • Edited • 1 week ago • Edited • Visible to...,2026-03-05,16,1,0,Shayak Mazumder,2026-03-12T22:59:02.061984
2,32b39a8e544e5b,Honored to be welcomed into the IAMAI (Interne...,Adya,"2,594 followers2,594 followers",,,https://www.linkedin.com/feed/update/urn:li:ac...,1mo • 1 month ago • Visible to anyone on or of...,2026-02-10,3,0,0,Adya AI,2026-03-12T23:04:13.091626
3,5db179b2d48f6a,Sona Sathishkumar from Karpagam College of Eng...,Adya,"2,594 followers2,594 followers",https://www.linkedin.com/in/sona-sathishkumar,,https://www.linkedin.com/feed/update/urn:li:ac...,1mo • 1 month ago • Visible to anyone on or of...,2026-02-10,2,0,0,Adya AI,2026-03-12T23:03:49.408602
4,eff1b679401243,Love this reflection from Priya Dharshini at K...,Adya,"2,594 followers2,594 followers",https://www.linkedin.com/in/priya-dharshini-b9...,,https://www.linkedin.com/feed/update/urn:li:ac...,1mo • 1 month ago • Visible to anyone on or of...,2026-02-10,3,0,0,Adya AI,2026-03-12T23:03:25.815868


## Cell 07 — Store to SQLite

In [7]:

def init_db(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path))
    conn.execute("""
        CREATE TABLE IF NOT EXISTS linkedin_posts (
            post_id             TEXT PRIMARY KEY,
            post_text           TEXT,
            author_name         TEXT,
            author_headline     TEXT,
            author_profile_url  TEXT,
            author_followers    TEXT,
            post_url            TEXT UNIQUE,
            date_posted         TEXT,
            date_parsed         TEXT,
            reactions           INTEGER DEFAULT 0,
            comments            INTEGER DEFAULT 0,
            reposts             INTEGER DEFAULT 0,
            search_term         TEXT,
            extracted_at        TEXT
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_date ON linkedin_posts (date_parsed)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_term ON linkedin_posts (search_term)")
    conn.commit()
    return conn


def upsert_posts(conn: sqlite3.Connection, df: pd.DataFrame) -> int:
    """Insert new posts; skip existing ones (INSERT OR IGNORE)."""
    inserted = 0
    for _, row in df.iterrows():
        try:
            conn.execute("""
                INSERT OR IGNORE INTO linkedin_posts
                (post_id, post_text, author_name, author_headline,
                 author_profile_url, author_followers, post_url,
                 date_posted, date_parsed, reactions, comments,
                 reposts, search_term, extracted_at)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
            """, (
                row.get("post_id",            ""),
                row.get("post_text",           ""),
                row.get("author_name",         ""),
                row.get("author_headline",     ""),
                row.get("author_profile_url",  ""),
                row.get("author_followers",    ""),
                row.get("post_url",            ""),
                row.get("date_posted",         ""),
                row.get("date_parsed",         ""),
                int(row.get("reactions",  0)),
                int(row.get("comments",   0)),
                int(row.get("reposts",    0)),
                row.get("search_term",         ""),
                row.get("extracted_at",        "")
            ))
            inserted += conn.execute("SELECT changes()").fetchone()[0]
        except Exception as e:
            print(f"  ⚠️  Row insert failed: {e}")
    conn.commit()
    return inserted


# ── Run ───────────────────────────────────────────────────────────────────────
conn = init_db(DB_PATH)
new_rows = upsert_posts(conn, df)
total_rows = conn.execute("SELECT COUNT(*) FROM linkedin_posts").fetchone()[0]

print(f"✅ Cell 07 complete")
print(f"   New rows inserted : {new_rows}")
print(f"   Total in database : {total_rows}")
print(f"   DB path           : {DB_PATH.resolve()}")

✅ Cell 07 complete
   New rows inserted : 3
   Total in database : 26
   DB path           : D:\Adya Techology\LinkedinScrapper\output\linkedin_posts.db


## Cell 08 — Export to JSON

In [8]:
# Read full dataset from SQLite
df_full = pd.read_sql_query(
    "SELECT * FROM linkedin_posts ORDER BY date_parsed DESC",
    conn
)

# Build structured JSON output
output = {
    "metadata": {
        "exported_at"    : datetime.now().isoformat(),
        "search_terms"   : SEARCH_TERMS,
        "date_range"     : {
            "from": CUTOFF_DATE.strftime("%Y-%m-%d"),
            "to"  : datetime.now().strftime("%Y-%m-%d")
        },
        "total_posts"    : len(df_full),
        "by_search_term" : df_full["search_term"].value_counts().to_dict(),
    },
    "posts": df_full.to_dict(orient="records")
}

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

file_size = JSON_PATH.stat().st_size / 1024
print(f"✅ Cell 08 complete")
print(f"   JSON saved → {JSON_PATH.resolve()}")
print(f"   File size  : {file_size:.1f} KB")
print(f"   Posts      : {len(df_full)}")
print(f"\nPreview (first post):")
if output["posts"]:
    for k, v in list(output["posts"][0].items())[:8]:
        print(f"  {k:25s}: {str(v)[:80]}")

✅ Cell 08 complete
   JSON saved → D:\Adya Techology\LinkedinScrapper\output\linkedin_mentions.json
   File size  : 38.5 KB
   Posts      : 26

Preview (first post):
  post_id                  : bb8a401c612d72
  post_text                : Adya AI is hiring for interns !!
  author_name              : Mahesh siddamsetty
  author_headline          : Product Management Leader - AI | B2B SaaS (0 to 1) | Global GTM Strategy (APAC, 
  author_profile_url       : https://www.linkedin.com/in/maheshsiddamsetty
  author_followers         : 
  post_url                 : https://www.linkedin.com/feed/update/urn:li:activity:7437094938820886528/
  date_posted              : 21h • 21 hours ago • Visible to anyone on or off LinkedIn


## Cell 09 — Export to Google Sheets

In [9]:
import gspread
from google.oauth2.service_account import Credentials
from pathlib import Path
 
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
 
# ── STEP 1: Diagnose the service account ─────────────────────────────────────
print("🔍 Diagnosing service account...")
 
if not Path(SA_JSON_PATH).exists():
    print(f"❌ service_account.json not found at: {SA_JSON_PATH}")
else:
    import json
    with open(SA_JSON_PATH) as f:
        sa_info = json.load(f)
    sa_email = sa_info.get("client_email", "NOT FOUND")
    print(f"✅ Service account file found")
    print(f"   Service account email : {sa_email}")
    print(f"\n{'='*60}")
    print(f"IMPORTANT: You must share your Google Sheet with this email:")
    print(f"  👉  {sa_email}")
    print(f"  Role: Editor")
    print(f"{'='*60}\n")
 
 
# ── STEP 2: The fixed push function ──────────────────────────────────────────
def push_to_google_sheets_fixed(df, sheet_name, sa_json):
    """
    FIXED version — does NOT try to create the sheet via API.
    Instead, YOU create the sheet manually and share it with the service account.
    This avoids all Drive permission issues.
    """
 
    # Auth
    creds  = Credentials.from_service_account_file(sa_json, scopes=SCOPES)
    client = gspread.authorize(creds)
    print("✅ Authenticated with Google")
 
    # Try to open the sheet
    try:
        spreadsheet = client.open(sheet_name)
        print(f"✅ Opened sheet: '{sheet_name}'")
    except gspread.SpreadsheetNotFound:
        print(f"\n❌ Sheet '{sheet_name}' not found.")
        print("   The service account cannot see this sheet.")
        print("\n   ── FIX: Do these 3 steps ──────────────────────────────────")
        print("   1. Go to sheets.google.com")
        print(f"   2. Create a new sheet named EXACTLY:  {sheet_name}")
        print(f"   3. Click Share → paste this email → set Editor → click Share:")
        print(f"      👉  {sa_email}")
        print("   4. Re-run this cell")
        print("   ─────────────────────────────────────────────────────────────")
        return None
    except Exception as e:
        err = str(e)
        if "403" in err:
            print(f"\n❌ Permission error (403).")
            print("   The service account email does NOT have access to the sheet.")
            print("\n   ── FIX ─────────────────────────────────────────────────")
            print("   1. Open your Google Sheet")
            print(f"   2. Click Share → add this email as Editor:")
            print(f"      👉  {sa_email}")
            print("   3. Re-run this cell")
            print("   ────────────────────────────────────────────────────────")
        else:
            print(f"\n❌ Unexpected error: {err}")
        return None
 
    # Clear and rewrite
    worksheet = spreadsheet.sheet1
    worksheet.clear()
    print("   Sheet cleared")
 
    # Write header
    headers = df.columns.tolist()
    worksheet.append_row(headers, value_input_option="RAW")
 
    # Format header row
    try:
        end_col = chr(64 + len(headers)) if len(headers) <= 26 else "N"
        worksheet.format(f"A1:{end_col}1", {
            "textFormat": {
                "bold": True,
                "foregroundColor": {"red": 1.0, "green": 1.0, "blue": 1.0}
            },
            "backgroundColor": {"red": 0.118, "green": 0.251, "blue": 0.690},
            "horizontalAlignment": "CENTER"
        })
    except Exception:
        pass  # formatting is cosmetic, skip if it fails
 
    # Write data in batches
    data  = df.fillna("").astype(str).values.tolist()
    BATCH = 500
    total = len(data)
 
    for i in range(0, total, BATCH):
        batch = data[i : i + BATCH]
        worksheet.append_rows(batch, value_input_option="RAW")
        done = min(i + BATCH, total)
        print(f"   Written {done}/{total} rows", end="\r")
 
    print(f"\n   Written {total}/{total} rows ✅")
 
    # Freeze header
    try:
        spreadsheet.sheet1.freeze(rows=1)
    except Exception:
        pass
 
    url = f"https://docs.google.com/spreadsheets/d/{spreadsheet.id}"
    return url
 
 
# ── STEP 3: Run ───────────────────────────────────────────────────────────────
print("📤 Pushing data to Google Sheets...")
result = push_to_google_sheets_fixed(df_full, SHEET_NAME, SA_JSON_PATH)
 
if result:
    print(f"\n✅ Cell 09 complete!")
    print(f"   Rows written : {len(df_full)}")
    print(f"   Open here    : {result}")
else:
    print("\n⚠️  Google Sheets export skipped.")
    print("   Your JSON file in output/linkedin_mentions.json has all the data.")
 

🔍 Diagnosing service account...
✅ Service account file found
   Service account email : linkedin-scraper@linkedin-scraper-489918.iam.gserviceaccount.com

IMPORTANT: You must share your Google Sheet with this email:
  👉  linkedin-scraper@linkedin-scraper-489918.iam.gserviceaccount.com
  Role: Editor

📤 Pushing data to Google Sheets...
✅ Authenticated with Google
✅ Opened sheet: 'AdyaTech'
   Sheet cleared
   Written 26/26 rows
   Written 26/26 rows ✅

✅ Cell 09 complete!
   Rows written : 26
   Open here    : https://docs.google.com/spreadsheets/d/1Qa1LOA4vCmdbAOuejXtOkU9wy6qf67r_yDaeNBCJWKQ


## Cell 10 — Interactive Query Tool

In [10]:
def query_posts(
    keyword      : str  = "",
    author       : str  = "",
    search_term  : str  = "",   # "Shayak Mazumder" | "Adya AI" | ""
    date_from    : str  = "",   # YYYY-MM-DD
    date_to      : str  = "",   # YYYY-MM-DD
    min_reactions: int  = 0,
    limit        : int  = 50
) -> pd.DataFrame:
    """
    Query the SQLite database with flexible filters.
    All parameters are optional — call with no args to get all posts.
    """
    conditions = []
    params     = []

    if keyword:
        conditions.append("post_text LIKE ?")
        params.append(f"%{keyword}%")
    if author:
        conditions.append("author_name LIKE ?")
        params.append(f"%{author}%")
    if search_term:
        conditions.append("search_term = ?")
        params.append(search_term)
    if date_from:
        conditions.append("date_parsed >= ?")
        params.append(date_from)
    if date_to:
        conditions.append("date_parsed <= ?")
        params.append(date_to)
    if min_reactions:
        conditions.append("reactions >= ?")
        params.append(min_reactions)

    where = "WHERE " + " AND ".join(conditions) if conditions else ""
    sql   = f"SELECT * FROM linkedin_posts {where} ORDER BY date_parsed DESC LIMIT ?"
    params.append(limit)

    return pd.read_sql_query(sql, conn, params=params)


# ── Summary Report ────────────────────────────────────────────────────────────
all_df = query_posts(limit=10000)
print("━" * 55)
print("  📊  LINKEDIN MENTIONS SUMMARY REPORT")
print("━" * 55)
print(f"  Total posts collected   : {len(all_df)}")
print(f"  Date range              : {all_df['date_parsed'].min()} → {all_df['date_parsed'].max()}")
print()
print("  By search term:")
for term, cnt in all_df["search_term"].value_counts().items():
    print(f"    {'•'} {term:<25} {cnt} posts")
print()
print("  Engagement totals:")
print(f"    Total reactions  : {all_df['reactions'].sum():,}")
print(f"    Total comments   : {all_df['comments'].sum():,}")
print(f"    Total reposts    : {all_df['reposts'].sum():,}")
print()
print("  Top 5 most-reacted posts:")
top5 = all_df.nlargest(5, "reactions")[["author_name", "reactions", "date_parsed", "post_url"]]
print(top5.to_string(index=False))
print("━" * 55)
print()

# ── Example queries ───────────────────────────────────────────────────────────
print("EXAMPLE QUERIES — edit and re-run this cell:")
print()
print("# All Adya AI posts:")
print('  query_posts(search_term="Adya AI")')
print()
print("# Posts with >50 reactions:")
print('  query_posts(min_reactions=50)')
print()
print("# Posts mentioning 'funding':")
print('  query_posts(keyword="funding")')
print()
print("# Posts from a specific author:")
print('  query_posts(author="Priya")')
print()
print("# Posts in January 2025:")
print('  query_posts(date_from="2025-01-01", date_to="2025-01-31")')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊  LINKEDIN MENTIONS SUMMARY REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total posts collected   : 26
  Date range              : 2025-10-12 → 2026-03-10

  By search term:
    • Shayak Mazumder           16 posts
    • Adya AI                   10 posts

  Engagement totals:
    Total reactions  : 486
    Total comments   : 23
    Total reposts    : 3

  Top 5 most-reacted posts:
    author_name  reactions date_parsed                                                                  post_url
    Akhil Jajoo         44  2025-10-12  https://www.linkedin.com/feed/update/urn:li:activity:7379920712075112449
    Neha Sharma         40  2025-12-11 https://www.linkedin.com/feed/update/urn:li:activity:7401118207895580672/
           Adya         38  2025-11-11 https://www.linkedin.com/feed/update/urn:li:activity:7383111026982006784/
  Athul Raj K K         37  2026-03-07 https://www.linkedin.com/feed/update/urn:li:ac

## Cell 11 — Close Browser

In [11]:
try:
    driver.quit()
    print("✅ Chrome closed")
except Exception:
    pass

try:
    conn.close()
    print("✅ SQLite connection closed")
except Exception:
    pass

print("\n🎉 Session complete. Output files:")
for f in OUTPUT_DIR.iterdir():
    print(f"   {f.name}  ({f.stat().st_size/1024:.1f} KB)")

✅ Chrome closed
✅ SQLite connection closed

🎉 Session complete. Output files:
   linkedin_mentions.json  (38.5 KB)
   linkedin_posts.db  (60.0 KB)
